# YoloMT on a Colab T4

YoloMT is a 449.8M-parameter streaming video captioner. Every second it writes a 30-50-word caption of the last 16 s
from the current frame and L cached past frames, with the lookback L chosen at run time from 0 to 16.

[Paper](https://arxiv.org/abs/2611.34601) | [Code](https://github.com/NessimBenA/YoloMT) | [Model](https://huggingface.co/Nessim/YoloMT) | [Dataset](https://huggingface.co/datasets/Nessim/YoloMT-labels)

To run it, pick the GPU with **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.

This notebook runs the PyTorch reference code of `scripts/caption.py`, which encodes the past frames again for every
caption, and prints the time each caption takes. The streaming engine in `deploy/` encodes each frame once and runs
the model in real time on a T4 with TensorRT.

## Install

The package at commit `9ec291a` of the code, with the versions of the Hugging Face libraries the model was trained
with. Colab's PyTorch stays: the package needs torch 2.8 or later.

In [ ]:
%pip install -q "yolomt @ git+https://github.com/NessimBenA/YoloMT@9ec291ae33fa543e790abd693e9d392a9f973c23" transformers==5.17.0 tokenizers==0.23.2 peft==0.21.0 safetensors==0.8.0 huggingface_hub==1.32.0 hf-xet==1.6.0 av==18.1.1

## Load the model

The weights of Nessim/YoloMT, and the configuration and tokenizer of its base model LiquidAI/LFM2.5-VL-450M, come from
the Hugging Face Hub. A GPU with native bfloat16 (compute capability 8.0 or higher) runs the model in bfloat16, as
`scripts/caption.py` does; the T4 (compute capability 7.5) runs it in float16, where some captions differ in wording
from the bfloat16 ones.

In [ ]:
import base64
import html
import importlib
import json
import sys
import time
import urllib.request
from pathlib import Path
from typing import Final, Protocol, TypedDict, cast

import numpy as np
import torch
from IPython.display import HTML, display

from yolomt.model import folder_of, load_folder, read_config, write
from yolomt.video import decode, duration, rgb

assert torch.cuda.is_available(), "No GPU: pick one with Runtime > Change runtime type > T4 GPU."
DEVICE: Final = torch.device("cuda")
CAPABILITY: Final = torch.cuda.get_device_capability(DEVICE)
DTYPE: Final = torch.bfloat16 if CAPABILITY >= (8, 0) else torch.float16
FOLDER: Final = folder_of("Nessim/YoloMT")
CONFIG: Final = read_config(FOLDER)
STUDENT: Final = load_folder(FOLDER, dtype=DTYPE).to(DEVICE).eval()
print(f"GPU: {torch.cuda.get_device_name(DEVICE)}, compute capability {CAPABILITY[0]}.{CAPABILITY[1]}, dtype: {DTYPE}")

## Captioning

`caption_video` makes the calls of `scripts/caption.py` with one window per forward pass: at second t it reads frame t
and the min(L, t) frames before it and writes a caption with greedy decoding under autocast. The frames are prepared as
for training: the first frame at or after each whole second, squashed to 256 x 256. `show` lists each second's frame
and caption, and `save` writes the captions to the working directory as WebVTT subtitles and as JSONL.

In [ ]:
class Caption(TypedDict):
    second: int
    caption: str


class ColabFiles(Protocol):
    def upload(self) -> dict[str, bytes]: ...


class Display(Protocol):
    def __call__(self, *objs: object) -> None: ...


class HtmlFactory(Protocol):
    def __call__(self, data: str) -> object: ...


def show_html(markup: str) -> None:
    cast(Display, display)(cast(HtmlFactory, HTML)(markup))


def frames_of(data: bytes, seconds: int) -> tuple[torch.Tensor, list[bytes]]:
    jpegs = decode(data, seconds=set(range(seconds)), grid=set()).student
    count = next((s for s in range(seconds) if s not in jpegs), seconds)
    assert count > 0, "The video has no frame to caption: it must last at least 1 s."
    array = np.stack([rgb(jpegs[s]) for s in range(count)])
    return torch.from_numpy(array).permute(0, 3, 1, 2).contiguous(), [jpegs[s] for s in range(count)]


def caption_video(frames: torch.Tensor, lookback: int) -> tuple[list[Caption], list[float]]:
    captions: list[Caption] = []
    times: list[float] = []
    for t in range(len(frames)):
        start = time.perf_counter()
        loops = min(lookback, t)
        pasts = torch.stack([frames[t - loops:t].flip(0)])
        with torch.autocast(DEVICE.type, dtype=DTYPE):
            texts = write(STUDENT, frames[[t]].to(DEVICE), pasts.to(DEVICE), loops=loops, max_new=CONFIG.max_new)
        times.append(time.perf_counter() - start)
        captions.append({"second": t, "caption": texts[0]})
        print(f"{t:4d} s  L = {loops:2d}  {times[-1]:5.2f} s  {texts[0]}")
    return captions, times


def show(captions: list[Caption], thumbnails: list[bytes], times: list[float], lookback: int) -> None:
    rows = "".join(
        f"<tr><td>{row['second']} s</td>"
        f"<td><img width='128' src='data:image/jpeg;base64,{base64.b64encode(jpeg).decode()}'></td>"
        f"<td style='text-align:left'>{html.escape(row['caption'])}</td></tr>"
        for row, jpeg in zip(captions, thumbnails))
    summary = (f"<p>{len(captions)} captions at L = {lookback} on {torch.cuda.get_device_name(DEVICE)} in {DTYPE}: "
               f"{sum(times) / len(times):.2f} s per caption on average.</p>")
    show_html(f"{summary}<table><tr><th>second</th><th>frame</th><th>caption</th></tr>{rows}</table>")


def stamp(seconds: int) -> str:
    return f"{seconds // 3600:02d}:{seconds // 60 % 60:02d}:{seconds % 60:02d}.000"


def save(name: str, captions: list[Caption]) -> None:
    vtt = Path(f"{name}.vtt")
    jsonl = Path(f"{name}.jsonl")
    vtt.write_text("WEBVTT\n\n" + "".join(
        f"{stamp(row['second'])} --> {stamp(row['second'] + 1)}\n{html.escape(row['caption'], quote=False)}\n\n"
        for row in captions))
    jsonl.write_text("".join(json.dumps(row) + "\n" for row in captions))
    print(f"Wrote {vtt} and {jsonl}")

## Example: a rocket launch

The clip is [Cygnus capsule lifted off atop an Antares rocket](https://commons.wikimedia.org/wiki/File:Cygnus_capsule_lifted_off_atop_an_Antares_rocket.webm),
a 45 s video by NASA in the public domain (PD-USGov-NASA), from Wikimedia Commons. `LOOKBACK` sets L, from 0 to 16.

In [ ]:
CLIP_URL: Final = "https://upload.wikimedia.org/wikipedia/commons/3/35/Cygnus_capsule_lifted_off_atop_an_Antares_rocket.webm"
CLIP_PAGE: Final = "https://commons.wikimedia.org/wiki/File:Cygnus_capsule_lifted_off_atop_an_Antares_rocket.webm"
CLIP_PATH: Final = Path("videos/antares-launch.webm")
LOOKBACK: int = 16

if not CLIP_PATH.exists():
    CLIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(CLIP_URL, headers={"User-Agent": "video-captioning-notebook/1.0"})
    with urllib.request.urlopen(request) as response:
        CLIP_PATH.write_bytes(response.read())
show_html(f"<p>Clip: <a href='{CLIP_PAGE}'>Cygnus capsule lifted off atop an Antares rocket</a> by NASA, "
          "public domain (PD-USGov-NASA), from Wikimedia Commons.</p>")
clip = CLIP_PATH.read_bytes()
frames, thumbnails = frames_of(clip, int(duration(clip)))
captions, times = caption_video(frames, LOOKBACK)
show(captions, thumbnails, times, LOOKBACK)
save("antares-launch", captions)

## Your own video

On Colab, the next cell opens a file picker, then captions the first `UPLOAD_SECONDS` seconds of each video you pick
at L = `UPLOAD_LOOKBACK`. Outside Colab it prints how to use it and does nothing else.

In [ ]:
UPLOAD_SECONDS: int = 40
UPLOAD_LOOKBACK: int = 16

if "google.colab" in sys.modules:
    colab_files = cast(ColabFiles, importlib.import_module("google.colab.files"))
    for name, data in colab_files.upload().items():
        length = duration(data)
        if int(length) > UPLOAD_SECONDS:
            print(f"{name} lasts {length:.1f} s: captioning its first {UPLOAD_SECONDS} s.")
        own_frames, own_thumbnails = frames_of(data, min(int(length), UPLOAD_SECONDS))
        own_captions, own_times = caption_video(own_frames, UPLOAD_LOOKBACK)
        show(own_captions, own_thumbnails, own_times, UPLOAD_LOOKBACK)
        save(Path(name).stem, own_captions)
else:
    print(f"Not on Colab: this cell opens a file picker on Colab and captions the first {UPLOAD_SECONDS} s of each "
          "video. Elsewhere, read a video with frames_of(Path(...).read_bytes(), seconds), then pass the frames to "
          "caption_video, show and save.")

## Files

Each run writes `<name>.vtt`, subtitles with one cue per second, and `<name>.jsonl`, one `{"second", "caption"}`
line per second as `scripts/caption.py` writes them, to the working directory. On Colab, download them from the Files
panel on the left.